Comparing the different approaches to eactract answers:
- how many invalid answers do we get with different regexes?
- check the invalid answers: why are they invalid?
- only look at the last 500 characters or so, is that enough to get the final answer? 
- how do we catch valid answers that are misaligned? the reported letter is not the same as the content? we could pass this to a language model after stripping out all the `\boxed` instances

In [139]:
from pathlib import Path
import pandas as pd
import yaml
from tqdm import tqdm
import re
import textwrap as tw

In [140]:
outputs = list(Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/comparison').rglob('*output.jsonl'))

In [63]:
data.benchmark_name.unique()

array(['medmcqa', 'test_pet', 'medqa_test', 'clinical_knowledge',
       'professional_medicine', 'USMLE_step3', 'USMLE_step2', 'anatomy',
       'test_cog', 'test_np_mixed', 'test_np_one', 'test_etpr', 'test_np',
       'medexpqa', 'USMLE_step1', 'USMLE_ethics'], dtype=object)

In [107]:
allowed_models = ['Qwen2.5-7B-Instruct','HuatuoGPT-o1-8B','Qwen2.5-3B-Instruct','Qwen3-4B','Qwen2.5-3B-DrGRPO-Strat-NACC']
allowed_benchmarks = [
    # 'medmcqa', 
    'test_pet', 
    'medqa_test', 
    'clinical_knowledge',
    #    'professional_medicine',   
       'test_cog', 
       'test_np_mixed', 
       'test_np_one', 
    #    'test_etpr', 
    #    'test_np',
       'medexpqa'
       ]

dfs = []
for file in tqdm(outputs):

    with open(file.parent / "config.yml") as config:
        config_dict = yaml.safe_load(config)

    if config_dict['run_readable_name'] in allowed_models and file.parent.parent.stem in allowed_benchmarks: 

        df = (
            pd.read_json(file, lines=True)
            .explode(column=["generated_text", "finish_reason"], ignore_index=True)
            .assign(
                model=config_dict["run_readable_name"],
                benchmark_name=file.parent.parent.stem
            )
        )

        dfs.append(df)

data = pd.concat(dfs,ignore_index=True)

del dfs

100%|██████████| 205/205 [00:09<00:00, 21.69it/s]


- make a boolean column with 'is the answer valid according to this extraction method?'
- when do we decide what the available options are? not all benchmarks have them sorted nicely. The nacc based ones do, we could do some preprocessing to add a string like 'ABCD' to the benchmark metadata (or a column) to make sure they are only extracting the right ones. It should actually be question specific because each question may have a different number of options

In [184]:
def print_example(df):
    print(tw.fill(df.sample(1)['generated_text'].item()[-600:],80))

def is_valid_last_boxed(text):
    # The pattern after the opening bracket is lazy, it finds as few character as 
    # possible until you hit a single letter (surrounded by word boundaries)
    # this means that if the output is \boxed{A and B} it will pick out A
    all_matches = re.findall(r"\\boxed\{.*?\b([A-Z0-9])\b", text)

    # finding more than one match is ambiguous, mark that as invalid too
    if len(all_matches) == 0: #or len(all_matches) > 1:
        # return "invalid"
        return False

    return True
    # return all_matches[-1].strip().upper()

def is_valid_last_boxed_no_think(text):
    # The pattern after the opening bracket is lazy, it finds as few character as 
    # possible until you hit a single letter (surrounded by word boundaries)
    # this means that if the output is \boxed{A and B} it will pick out A
    all_matches = re.findall(r"\\boxed\{.*?\b([A-Z0-9])\b", remove_think(text))

    # finding more than one match is ambiguous, mark that as invalid too
    if len(all_matches) == 0: #or len(all_matches) > 1:
        # return "invalid"
        return False

    return True
    # return all_matches[-1].strip().upper()

def is_valid_only_boxed(text):
    # The pattern after the opening bracket is lazy, it finds as few character as 
    # possible until you hit a single letter (surrounded by word boundaries)
    # this means that if the output is \boxed{A and B} it will pick out A
    all_matches = re.findall(r"\\boxed\{.*?\b([A-Z0-9])\b", text)

    # finding more than one match is ambiguous, mark that as invalid too
    if len(all_matches) == 0 or len(all_matches) > 1:
        # return "invalid"
        return False

    return True

def is_valid_only_boxed_unique(text):
    # The pattern after the opening bracket is lazy, it finds as few character as 
    # possible until you hit a single letter (surrounded by word boundaries)
    # this means that if the output is \boxed{A and B} it will pick out A
    all_matches = re.findall(r"\\boxed\{.*?\b([A-Z0-9])\b", text)

    # finding more than one match is ambiguous, mark that as invalid too
    if len(all_matches) == 0 or len(set(all_matches)) > 1:
        # return "invalid"
        return False

    return True
    # return all_matches[-1].strip().upper()

def is_valid_only_boxed_500(text):
    # The pattern after the opening bracket is lazy, it finds as few character as 
    # possible until you hit a single letter (surrounded by word boundaries)
    # this means that if the output is \boxed{A and B} it will pick out A
    all_matches = re.findall(r"\\boxed\{.*?\b([A-Z0-9])\b", text[-500:])

    # finding more than one match is ambiguous, mark that as invalid too
    if len(all_matches) == 0 or len(all_matches) > 1:
        # return "invalid"
        return False

    return True
    # return all_matches[-1].strip().upper()

def remove_think(text):
    # Greedily remove all content between think tags. It removes 
    # from the first think tags to the last, even if there are more tags in between
    stripped = re.sub(r"<think>.*</think>", "", text, flags=re.DOTALL).strip()
    # stripped = re.sub(r"## Thinking.*?(?=##)", "", stripped, flags=re.DOTALL).strip() # for huatuogpt

    return stripped

def is_valid_only_boxed_no_think(text):
    # The pattern after the opening bracket is lazy, it finds as few character as 
    # possible until you hit a single letter (surrounded by word boundaries)
    # this means that if the output is \boxed{A and B} it will pick out A
    all_matches = re.findall(r"\\boxed\{.*?\b([A-Z0-9])\b", remove_think(text))

    # finding more than one match is ambiguous, mark that as invalid too
    if len(all_matches) == 0 or len(all_matches) > 1:
        # return "invalid"
        return False

    return True
    # return all_matches[-1].strip().upper()

def is_valid_last_boxed_period(text):
    # The pattern after the opening bracket is lazy, it finds as few character as 
    # possible until you hit a single letter (surrounded by word boundaries)
    # this means that if the output is \boxed{A and B} it will pick out A
    all_matches = re.findall(r'\\boxed{([A-Z0-9])(?:\.\s*[^}]*)?}', text)

    # finding more than one match is ambiguous, mark that as invalid too
    if len(all_matches) == 0: #or len(all_matches) > 1:
        # return "invalid"
        return False

    return True
    # return all_matches[-1].strip().upper()

def is_valid_only_boxed_period(text):
    # The pattern after the opening bracket is lazy, it finds as few character as 
    # possible until you hit a single letter (surrounded by word boundaries)
    # this means that if the output is \boxed{A and B} it will pick out A
    all_matches = re.findall(r'\\boxed{([A-Z0-9])(?:\.\s*[^}]*)?}', text)

    # finding more than one match is ambiguous, mark that as invalid too
    if len(all_matches) == 0 or len(all_matches) > 1:
        # return "invalid"
        return False

    return True
    # return all_matches[-1].strip().upper()

In [185]:
data

,problem,generated_text,finish_reason,model,benchmark_name,valid_last_boxed,valid_last_boxed_no_think,valid_only_boxed,valid_only_boxed_no_think,valid_only_boxed_500,multiple_boxed,valid_only_boxed_unique,switch_answers,valid_last_boxed_period
0,"{'ID': 'NACC959224', 'patient_summary': '{ ...","## Thinking\n\nOkay, so we've got a 73-year-ol...",stop,HuatuoGPT-o1-8B,test_pet,False,False,False,False,False,False,False,False,False
1,"{'ID': 'NACC959224', 'patient_summary': '{ ...","## Thinking\n\nAlright, let's think about this...",stop,HuatuoGPT-o1-8B,test_pet,True,True,True,True,True,False,True,False,True
2,"{'ID': 'NACC959224', 'patient_summary': '{ ...","## Thinking\n\nAlright, so we've got a 73-year...",stop,HuatuoGPT-o1-8B,test_pet,True,True,True,True,True,False,True,False,True
3,"{'ID': 'NACC959224', 'patient_summary': '{ ...",## Thinking\n\nLet's think about this. The pat...,stop,HuatuoGPT-o1-8B,test_pet,True,True,True,True,True,False,True,False,True
4,"{'ID': 'NACC959224', 'patient_summary': '{ ...","## Thinking\n\nAlright, let's look at this sit...",stop,HuatuoGPT-o1-8B,test_pet,True,True,True,True,True,False,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259750,"{'ground_truth': 2, 'full_answer': 'Typical of...","## Thinking\n\nOkay, let's think this through....",stop,HuatuoGPT-o1-8B,medexpqa,False,False,False,False,False,False,False,False,False
259751,"{'ground_truth': 2, 'full_answer': 'Typical of...","## Thinking\n\nAlright, let's figure out what'...",stop,HuatuoGPT-o1-8B,medexpqa,False,False,False,False,False,False,False,False,False
259752,"{'ground_truth': 2, 'full_answer': 'Typical of...","## Thinking\n\nOkay, let's think about this 67...",stop,HuatuoGPT-o1-8B,medexpqa,False,False,False,False,False,False,False,False,False
259753,"{'ground_truth': 2, 'full_answer': 'Typical of...","## Thinking\n\nAlright, let's think about this...",stop,HuatuoGPT-o1-8B,medexpqa,True,True,True,True,True,False,True,False,False


In [186]:
data = data.assign(
    valid_last_boxed=data["generated_text"].apply(is_valid_last_boxed),
    # valid_last_boxed_no_think=data["generated_text"].apply(is_valid_last_boxed_no_think),
    valid_only_boxed=data["generated_text"].apply(is_valid_only_boxed),
    valid_only_boxed_unique=data["generated_text"].apply(is_valid_only_boxed_unique),
    # valid_only_boxed_no_think=data["generated_text"].apply(is_valid_only_boxed_no_think),
    # valid_only_boxed_500=data["generated_text"].apply(is_valid_only_boxed_500),
    valid_last_boxed_period=data["generated_text"].apply(is_valid_last_boxed_period),
    # valid_only_boxed_period=data["generated_text"].apply(is_valid_only_boxed_period),
)

# mask = data[
#     ["valid_last_boxed",
#     "valid_only_boxed",
#     "valid_last_boxed_period",
#     "valid_only_boxed_period",
#     ]
# ].nunique(axis=1) > 1


In [187]:
data.groupby(['benchmark_name','model']).agg(
    {
    'valid_last_boxed':lambda x: x.sum()/len(x),
    'valid_last_boxed_period':lambda x: x.sum()/len(x),
    # 'valid_last_boxed_no_think':lambda x: x.sum()/len(x),
    # 'valid_only_boxed':lambda x: x.sum()/len(x),
    # 'valid_only_boxed_unique':lambda x: x.sum()/len(x),
    # 'valid_only_boxed_no_think':lambda x: x.sum()/len(x),
    }
    ).sort_index(level=[1,0]).round(5)*100

,,valid_last_boxed,valid_last_boxed_period
benchmark_name,model,,
clinical_knowledge,HuatuoGPT-o1-8B,61.283,59.925
medexpqa,HuatuoGPT-o1-8B,44.298,44.132
medqa_test,HuatuoGPT-o1-8B,36.861,36.590
test_cog,HuatuoGPT-o1-8B,28.926,28.882
test_np_mixed,HuatuoGPT-o1-8B,15.662,15.662
test_np_one,HuatuoGPT-o1-8B,16.809,16.755
test_pet,HuatuoGPT-o1-8B,38.308,38.154
clinical_knowledge,Qwen2.5-3B-DrGRPO-Strat-NACC,99.698,98.717
medexpqa,Qwen2.5-3B-DrGRPO-Strat-NACC,99.504,98.017


These are cases where there are multiple instances of `boxed`, how many can we recover if we only keep the last few characters?

In [163]:
mask = data["valid_last_boxed"] & ~data["valid_only_boxed"]

data['multiple_boxed'] = mask

data.groupby(['benchmark_name','model']).agg({
    'multiple_boxed':lambda x: x.sum()/len(x),
    }).sort_index(level=[1,0])*100

,,multiple_boxed
benchmark_name,model,
clinical_knowledge,HuatuoGPT-o1-8B,0.301887
medexpqa,HuatuoGPT-o1-8B,0.000000
medqa_test,HuatuoGPT-o1-8B,0.159363
test_cog,HuatuoGPT-o1-8B,0.020647
test_np_mixed,HuatuoGPT-o1-8B,0.112676
test_np_one,HuatuoGPT-o1-8B,0.000000
test_pet,HuatuoGPT-o1-8B,0.000000
clinical_knowledge,Qwen2.5-3B-DrGRPO-Strat-NACC,0.754717
medexpqa,Qwen2.5-3B-DrGRPO-Strat-NACC,0.495868


In [179]:
# print(tw.fill(data[data.multiple_boxed & (data['benchmark_name'] == 'test_np_mixed')].sample(1)['generated_text'].item()[-500:],80))
print(tw.fill(data[data.multiple_boxed].sample(1)['generated_text'].item()[-600:],80))

's disease.  From the given information and reasoning, the patient seems to have
Alzheimer's disease pathology with an APOE e4 e4 genotype. There is no evidence
for Lewy Body dementia, and there is also a discrepancy in her presentation
indicating that FTLD tau and TDP-43 are not specific markers. Hence:  \boxed{E.
Alzheimer's disease pathology (AD) and Lewy body pathology (LBD)} is incorrect,
and one should look for pathologies specific to AD or LBD, which does not appear
to fit entirely.   The best option derived from the presented data would be
\boxed{E. Alzheimer's disease pathology (AD)}.


find where in the string the answer is?

In [165]:
# def find_all_boxed(text):
#     # The pattern after the opening bracket is lazy, it finds as few character as 
#     # possible until you hit a single letter (surrounded by word boundaries)
#     # this means that if the output is \boxed{A and B} it will pick out A
#     all_matches = re.findall(r"\\boxed\{.*?\b([A-Z0-9])\b", text)

#     # finding more than one match is ambiguous, mark that as invalid too
#     if len(all_matches) == 0 or len(all_matches) > 1:
#         # return "invalid"
#         return False

percent of multiple answers that are actually different

In [177]:
data['switch_answers'] = (data['valid_last_boxed']) & (~data['valid_only_boxed']) & (~data['valid_only_boxed_unique'])


data.groupby(['model','benchmark_name']).agg({'switch_answers': lambda x: x.sum()/len(x)})*100

switch_answers
model                        benchmark_name                    
HuatuoGPT-o1-8B              clinical_knowledge        0.301887
                             medexpqa                  0.000000
                             medqa_test                0.143426
                             test_cog                  0.020647
                             test_np_mixed             0.112676
                             test_np_one               0.000000
                             test_pet                  0.000000
Qwen2.5-3B-DrGRPO-Strat-NACC clinical_knowledge        0.452830
                             medexpqa                  0.165289
                             medqa_test                0.494024
                             test_cog                  0.067103
                             test_np_mixed             0.281690
                             test_np_one               0.132979
                             test_pet                  0.000000
Qwen2.5-3B-Instruct          clinical_knowledge        0.377358
                             medexpqa                  0.000000
                             medqa_test                0.541833
                             test_cog                  0.068823
                             test_np_mixed             0.957746
                             test_np_one               0.265957
                             test_pet                  0.000000
Qwen2.5-7B-Instruct          clinical_knowledge        0.075472
                             medexpqa                  0.330579
                             medqa_test                0.063745
                             test_cog                  0.006882
                             test_np_mixed             1.859155
                             test_np_one               0.106383
                             test_pet                  0.000000
Qwen3-4B                     clinical_knowledge        0.603774
                             medexpqa                  0.661157
                             medqa_test                0.063745
                             test_cog                  0.003441
                             test_np_mixed             0.450704
                             test_np_one               0.000000
                             test_pet                  0.000000

In [188]:
print_example(data[
    (data['valid_last_boxed']) &
    (~data['valid_only_boxed']) &
    (~data['valid_only_boxed_unique'])
])

maging findings, the pathologies underlying the patient's cognitive symptoms are
most likely due to:  \boxed{A. Alzheimer's disease pathology (AD), Lewy body
pathology (LBD) and Frontotemporal Lobar Degeneration with tau pathology or
TDP-43 pathology (FTLD)}   However, since AD and inclusion of FTLD are suggested
and LBD findings are not strongly supported, Alzheimer's disease pathology (AD)
could be the dominant one presented. Thus, the answer is best narrowed to:
\boxed{C. Alzheimer's disease pathology (AD) and Frontotemporal Lobar
Degeneration with tau pathology or TDP-43 pathology (FTLD)}


- do the answers change between the two different regexes? like, are they capturing different letters?
- for inconsistent multiple boxed, do the regex and llm agree?

In [158]:
# len(set(all_matches))
# all_matches = re.findall(r"\\boxed\{.*?\b([A-Z0-9])\b", text)

1